# Step 20.1 — Save Model Weights

本节学习 **parameter tree serialization（参数树序列化）** 与 **file organization（文件组织）**，沿用项目固定的 MLX v0.32.2 和已验收的完整 MiniGPT 定义。

19.5 最终材料提交 `059281799985a461994b396e16ebe9e60f3616cc` 已独立核心静态验收 PASS，累计修正 2/2；19.2、19.3 各为 1/2。这是材料验收，不是学员掌握或运行反馈。

本节初始生成 0/2。Notebook 中所有代码单元均未执行、无输出。本轮只准备材料，不创建真实权重文件；完成即停在 20.1，不进入 20.2。

## 1. 本节的保存对象：参数树

**Model weights（模型权重）**是模型用于前向计算的参数数组，例如 embedding、attention/MLP 的权重和 bias、RMSNorm 参数与位置 embedding。它们组织成嵌套的 **parameter tree（参数树）**，而不是一个单独的长向量。

在 MLX 中，`model.parameters()` 递归返回模型中数组参数的嵌套字典/列表结构；`model.state` 则是更广义的模块状态容器。本节使用专用的 `model.save_weights(path)` 保存权重文件，不直接把整个Python对象pickle起来。

重要边界：当前仓库没有由本轮生成或验证的训练权重。本节的setup仅用于未来工作副本中构造一个明确的新模型，若直接保存它，得到的是**初始化权重**，不是训练成果。

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn

# Configuration is explicit and self-contained; vocab_size must be supplied by the caller.
MODEL_CONFIG = {
    "vocab_size": None,  # Set to the real vocabulary size from the same character mapping.
    "dims": 64,
    "num_heads": 4,
    "num_blocks": 2,
    "max_seq_len": 64,
}

def make_model_kwargs(config):
    if not isinstance(config["vocab_size"], int) or config["vocab_size"] < 1:
        raise ValueError("Set vocab_size from the real token-ID mapping before model construction.")
    if config["dims"] % config["num_heads"] != 0:
        raise ValueError("dims must be divisible by num_heads.")
    return dict(config)

## 2. 恢复完整模型定义

以下定义直接沿用通过的 Step 15.4 母版：Causal Multi-Head Attention、Feed-Forward MLP、Transformer Block 和 MiniGPT。保存权重要求模型结构与创建时一致；本节不改动架构，也不以 toy 模型替代完整 MiniGPT。

In [ ]:
import math
import mlx.core as mx
import mlx.nn as nn


class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)  # (B,T,C)


class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# Definitions only. This master notebook does not execute them.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


# Definition only. No MiniGPT forward is executed in this master notebook.


## 3. Shape 与参数树如何对应？

本候选的 width (C=64)、heads (H=4)，所以每头宽度 (D=C/H=16)。若真实词表大小为 (V)，主要参数形状包括：

| 参数 | Shape |
|---|---|
| Token embedding | ((V,64)) |
| Position embedding | ((64,64)) |
| 每个 Block 的合并 QKV 权重 | ((192,64)) |
| Attention 输出投影 | ((64,64)) |
| MLP 第一层权重 | ((256,64)) |
| MLP 第二层权重 | ((64,256)) |
| LM head 权重 | ((V,64)) |

这些形状说明参数树里保存的是一组有名称、有结构的数组。实际参数总数取决于真实 (V) 与现有完整模型定义；本节不假定词表大小，也不生成具体权重文件。

可在未来工作副本中检查参数树的顶层键和某个权重的shape。此类检查是模型结构信息，不等于训练状态。

In [ ]:
# Future work-copy setup: provide the true vocab_size from the project's character mapping.
# This cell deliberately does not load data or infer a fake vocabulary size.
def inspect_parameter_tree(model):
    params = model.parameters()
    summary = {
        "top_level_keys": list(params.keys()),
        "embedding_shape": tuple(params["token_embedding"]["weight"].shape)
            if "token_embedding" in params else None,
    }
    return summary

# After setting MODEL_CONFIG["vocab_size"] from the real mapping, a future learner may run:
# model = MiniGPT(**make_model_kwargs(MODEL_CONFIG))
# mx.eval(model.parameters())
# print(inspect_parameter_tree(model))


## 4. MLX v0.32.2 的权重保存接口

固定版本的 `mlx.nn.Module.save_weights(file)` 根据文件扩展名选择格式：`.npz` 使用 `mx.savez()`，`.safetensors` 使用 `mx.save_safetensors()`。本节选择 **SafeTensors**，以便将模型权重保存为明确的参数数组集合。

保存前调用：

`mx.eval(model.parameters())`

是为了让待保存的参数数组及其依赖完成求值。它不会训练模型，也不会让随机初始化参数变成训练后权重。

官方固定版本依据：[MLX v0.32.2 Module API](https://github.com/ml-explore/mlx/blob/v0.32.2/python/mlx/nn/layers/base.py)；[MLX v0.32.2 saving/loading 文档](https://github.com/ml-explore/mlx/tree/v0.32.2/docs/src/usage)。

## 5. 安全组织输出目录，避免覆盖旧文件

本节使用 **UTC timestamp（协调世界时的时间戳）**建立独立目录，并通过 `mkdir(exist_ok=False)` 原子地拒绝已存在的目录。权重文件只写入刚成功创建的新目录，因此不会覆盖旧的同名权重文件。

目录结构示例（只是路径模板，不代表文件已存在）：

```text
results/
└── model_weights/
    └── minigpt_weights_YYYYMMDDTHHMMSSZ/
        └── model.safetensors
```

不要把该文件误认为完整 checkpoint。模型权重文件不自动包含 optimizer 的 Adam 一阶/二阶矩、step、训练 RNG、sampler RNG、数据 split、当前训练步数或全部运行配置。

In [ ]:
def save_weights_to_new_directory(model, project_root, run_label="minigpt_weights"):
    root = Path(project_root) / "results" / "model_weights"
    root.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_dir = root / f"{run_label}_{stamp}"
    # exist_ok=False prevents reusing an existing run directory.
    output_dir.mkdir(exist_ok=False)

    weights_path = output_dir / "model.safetensors"
    # Materialization boundary: finish parameter arrays before serialization.
    mx.eval(model.parameters())
    model.save_weights(str(weights_path))
    return weights_path

# Definition only. This function is not called in the published master.

## 6. 从真实模型到未来权重文件：显式调用顺序

本节不读取语料，因此不自动决定 (V)。未来练习者应从项目既有的同一字符表/token-ID 映射取得真实词表大小，再构造模型。

保存流程的逻辑顺序为：

1. 明确配置和真实 `vocab_size`。
2. 构造完整 MiniGPT。
3. 明确是否这是训练后的模型；如果不是，记录为初始化权重。
4. 完成参数求值。
5. 创建独立输出目录。
6. 调用 `model.save_weights(...)`。
7. 记录文件路径、配置与权重来源。

本节不伪造已有训练结果，也不在没有训练权重的情况下生成所谓 baseline checkpoint。

## 7. 权重文件不等于完整训练恢复

| 内容 | 单独的 `model.safetensors` 是否包含 |
|---|---|
| 模型参数数组 | 是 |
| 模型结构代码和构造参数 | 不保证 |
| Adam 的 m/v 与 step | 否 |
| Python/MLX RNG 状态 | 否 |
| 数据切分与 sampler 状态 | 否 |
| 训练步数与 loss history | 否 |
| 运行环境与超参数记录 | 不保证 |

因此，权重文件适合后续重新构造同样结构的模型并加载参数；要精确继续训练，还需额外保存匹配的优化器状态、随机状态、数据/采样位置和配置。完整恢复训练状态不是本节范围，留待后续课节。

MLX 的 `load_weights(..., strict=True)` 默认要求权重匹配模型参数结构。加载前应使用同样的模型架构与参数命名，不要将不同宽度、层数或词表的权重混用。

## 本节边界与累计计数

本节只准备参数树序列化、SafeTensors 文件组织和安全输出路径范式。所有保存代码只写入材料，未调用保存函数、未创建权重文件，也未执行 Notebook。

历史修正计数保持：16.1/16.2 各1/2；19.2/19.3各1/2；19.5为2/2；其余已验收课节沿用原计数。20.1本次初始生成0/2。停止在20.1，不进入20.2；后续按请求逐小节至大Step20全部材料完成后停止，不进入21。